<a href="https://colab.research.google.com/github/marziemajidi/AI/blob/LLM/Chapter_2_Tokens_and_Token_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1>Chapter 2 - Tokens and Token Embeddings</h1>
<i>Exploring tokens and embeddings as an integral part of building LLMs</i>


<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961"><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/HandsOnLLM/Hands-On-Large-Language-Models"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HandsOnLLM/Hands-On-Large-Language-Models/blob/main/chapter02/Chapter%202%20-%20Tokens%20and%20Token%20Embeddings.ipynb)

---

This notebook is for Chapter 2 of the [Hands-On Large Language Models](https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961) book by [Jay Alammar](https://www.linkedin.com/in/jalammar) and [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/).

---

<a href="https://www.amazon.com/Hands-Large-Language-Models-Understanding/dp/1098150961">
<img src="https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/images/book_cover.png" width="350"/></a>


### [OPTIONAL] - Installing Packages on <img src="https://colab.google/static/images/icons/colab.png" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** the following codeblock to install the dependencies for this chapter:

---

💡 **NOTE**: We will want to use a GPU to run the examples in this notebook. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**.

---

In [ ]:
# %%capture
# !pip install transformers>=4.41.2 sentence-transformers>=3.0.1 gensim>=4.3.2 scikit-learn>=1.5.0 accelerate>=0.31.0
!pip install transformers==4.33.2
!pip install torch==2.0.1

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.9/119.9 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 62.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 93.2 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.21.2
    Uninstalling tokenizers-0.21.2:
      Successfully uninstalled tokenizers-0.21.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.4
    Uninstalling transformers-4.52.4:
      Successfully uninstalled transformers-4.52.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 4.1.0 requires transformers<5.0.0,>=4.41.0, but you have transformers 4.33.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Downloading and Running An LLM

The first step is to load our model onto the GPU for faster inference. Note that we load the model and tokenizer separately and keep them as such so that we can explore them separately.

 This code is for loading and working **with a Large Language Model (LLM) using the Hugging Face transformers library**.

The Hugging Face transformers library allows you to use **pre-trained models like GPT, BERT, etc.**, to solve various natural language processing (NLP) tasks. In this case, you are using the **"microsoft/Phi-3-mini-4k-instruct" model, which is a causal language model designed for text generation.**

If you want to use the model on a GPU for faster computations, you also need to **install PyTorch**:

**AutoModelForCausalLM**: This class is used to load a causal language model, which **generates text based on a given prompt**.
AutoTokenizer: This class handles text processing,converting input text into tokens (numerical representations) and converting tokens back into readable text.

from_pretrained: T**his method downloads the specified pre-trained model from the Hugging Face model hub** and loads it for use.
Arguments:

    "microsoft/Phi-3-mini-4k-instruct": This is the name of the model. **It’s a pre-trained language model by Microsoft designed for generating instructions or completing tasks.**
    device_map="cuda": This specifies that **the model should run on a GPU.** If you don’t have a GPU, you can change this to "cpu".
    torch_dtype="auto": Automatically selects the best data type for the model (e.g., float16 on GPUs for faster inference).
    trust_remote_code=True: Some models have custom code in their repositories. This flag allows the model to use that custom code.

What is a Tokenizer?

    A tokenizer converts **text into tokens (numerical representations) that the model can understand.**
    It also converts the **model’s output (tokens) back into human-readable text.**

**from_pretrained**: This method **ensures that the tokenizer is compatible with the loaded model.**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
# Step 1: Load the pre-trained language model
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct",  # Name of the model
    device_map="cuda",                  # Run the model on GPU for faster computations
    torch_dtype="auto",                 # Automatically choose the data type (e.g., float16 for faster inference)
    trust_remote_code=True              # Allow the use of custom code from the model repository
)

# Step 2: Load the tokenizer associated with the model
tokenizer = AutoTokenizer.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct"  # Ensure the tokenizer matches the model
)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

configuration_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- configuration_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


modeling_phi3.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Phi-3-mini-4k-instruct:
- modeling_phi3.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


ModuleNotFoundError: No module named 'transformers.cache_utils'

**Why tokenize the input?**
Language models work only with numerical data, not raw text. The tokenizer converts the input text into numbers (token IDs) that the model can understand.

**tokenizer(prompt):**
This tokenizes the text stored in the prompt variable (the instruction or question you give to the model).

---


**return_tensors="pt":**
Converts the output into a **PyTorch-compatible tensor**, the standard format used by the model.

---


**.input_ids:**
Extracts only the **token IDs** (the numerical representation of tokens).


---


**.to("cuda"):**
**Moves the token IDs to the GPU** for faster processing, as the model runs on the GPU.

**What does generate do?**
This function tells the model to generate new text based on the input tokens.


---

Parameters:
    **input_ids:**
    These are the tokens generated in the previous step, passed as input to the model.
**max_new_tokens=30:**
    Limits the model to generate at most 30 new tokens. This prevents overly long outputs.

**Why decode the output?**
The model generates **its output as a sequence of token IDs**. To convert these **numerical tokens back into human-readable text**, we use **the decode function**.

Code breakdown:

    **generation_output[0]:**
    The model might produce multiple outputs, but here we take only the first one.
    **tokenizer.decode(...):**
    Converts the token IDs into a readable string.

In [ ]:
prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|>"

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate the text
generation_output = model.generate(
  input_ids=input_ids,
  max_new_tokens=30
)

# Print the output
print(tokenizer.decode(generation_output[0]))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Calling `get_max_cache()` will raise error from v4.48


Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened.<|assistant|> Subject: Sincere Apologies for the Gardening Mishap


Dear Sarah,


I hope this message finds


By default, the **.generate()** method produces only one output sequence unless you explicitly specify that you want multiple sequences using the **num_return_sequences** parameter.

---

Correctly access the first (and only) output in the generation_output, which is generation_output[0].
**If you want multiple outputs**, set **num_return_sequences** when generating.



In [ ]:
len(generation_output[0])

54

Looking at the code, we can see that the model does not in fact receive the
text prompt **bold text**. Instead, the tokenizers processed the input prompt, and returned
the information the model needed in the variable input_ids, which the
model used as its input.

Looking at the code, we can see that the model does not in fact receive the
text prompt. **bold text** Instead, the **tokenizers processed the input prompt**, and returned
the information the **model needed in the variable input_ids,** which the
**model used as its input.**
Let’s print input_ids to see what it holds inside:

In [ ]:
print(input_ids)

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001]], device='cuda:0')


In [ ]:
for id in input_ids[0]:
   print(id)

tensor(14350, device='cuda:0')
tensor(385, device='cuda:0')
tensor(4876, device='cuda:0')
tensor(27746, device='cuda:0')
tensor(5281, device='cuda:0')
tensor(304, device='cuda:0')
tensor(19235, device='cuda:0')
tensor(363, device='cuda:0')
tensor(278, device='cuda:0')
tensor(25305, device='cuda:0')
tensor(293, device='cuda:0')
tensor(16423, device='cuda:0')
tensor(292, device='cuda:0')
tensor(286, device='cuda:0')
tensor(728, device='cuda:0')
tensor(481, device='cuda:0')
tensor(29889, device='cuda:0')
tensor(12027, device='cuda:0')
tensor(7420, device='cuda:0')
tensor(920, device='cuda:0')
tensor(372, device='cuda:0')
tensor(9559, device='cuda:0')
tensor(29889, device='cuda:0')
tensor(32001, device='cuda:0')


This reveals the inputs that **LLMs respond to, a series of integers** as shown in
Figure 2-4. **Each one is the unique ID for a specific token (character, word,**
or part of a word) . These IDs reference a table inside the tokenizer
containing all the tokens it knows.

In [ ]:
for id in input_ids[0]:
   print(tokenizer.decode(id))

Write
an
email
apolog
izing
to
Sarah
for
the
trag
ic
garden
ing
m
ish
ap
.
Exp
lain
how
it
happened
.
<|assistant|>


In [ ]:
print(tokenizer.decode(1))
print(tokenizer.decode(2))
print(tokenizer.decode([3323, 622]))
print(tokenizer.decode(29901))

<s>
</s>
Subject
:


we can also inspect **the tokens generated by the model** by
printing the **generation_output** variable. This shows the **input tokens**
*as well as* the **output tokens**

In [ ]:
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   304, 19235,   363,   278, 25305,
           293, 16423,   292,   286,   728,   481, 29889, 12027,  7420,   920,
           372,  9559, 29889, 32001,  3323,   622, 29901,   317,  3742,   406,
          6225, 11763,   363,   278, 19906,   292,   341,   728,   481,    13,
            13,    13, 29928,   799, 19235, 29892,    13,    13,    13, 29902,
          4966,   445,  2643, 14061]], device='cuda:0')

we need the **tokenizer on the output side to translate the**
**token ID into the actual text**. We do that using the tokenizer’s decode method.
We can pass it an individual token ID or a list of them:

In [ ]:
tokenizer.decode(14350)

'Write'

# Comparing Trained LLM Tokenizers


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

colors_list = [
    '102;194;165', '252;141;98', '141;160;203',
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' +
            tokenizer.decode(t) +
            '\x1b[0m',
            end=' '
        )

In [ ]:
from transformers import AutoTokenizer

# Define a list of RGB color codes for token background coloring
colors_list = [
    '102;194;165',  # Light green
    '252;141;98',   # Light orange
    '141;160;203',  # Light blue
    '231;138;195',  # Light pink
    '166;216;84',   # Light lime green
    '255;217;47'    # Bright yellow
]

# Function to tokenize a sentence and display each token with a colored background
def show_tokens2(sentence, tokenizer_name):
    # Load the tokenizer for the specified model from Hugging Face Transformers
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    # Tokenize the sentence and get the list of token IDs
    token_ids = tokenizer(sentence).input_ids

    # Loop through each token and its index
    for idx, token_id in enumerate(token_ids):
        # Calculate the background color for the current token using the index
        background_color = colors_list[idx % len(colors_list)]

        # Decode the token ID into its text representation
        decoded_token = tokenizer.decode(token_id)

        # Format the token with the background color and reset styles after
        colored_token = f'\x1b[0;30;48;2;{background_color}m{decoded_token}\x1b[0m'

        # Print the colored token without breaking the line
        print(colored_token, end=' ')


In [ ]:
text = """
English and CAPITALIZATION
🎵 鸟
show_tokens False None elif == >= else: two tabs:"    " Three tabs: "       "
12.0*50=600
"""

In [ ]:
show_tokens2(text, "bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

[CLS] english and capital ##ization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 [SEP] 

In [ ]:
show_tokens(text, "bert-base-cased")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

[CLS] English and CA ##PI ##TA ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 [SEP] 

In [ ]:
show_tokens(text, "gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]


 English  and  CAP ITAL IZ ATION 
 � � �  � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"        "  Three  tabs :  "              " 
 12 . 0 * 50 = 600 
 

In [ ]:
show_tokens(text, "google/flan-t5-small")

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

English and CA PI TAL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600  </s> 

In [ ]:
# The official is `tiktoken` but this the same tokenizer on the HF platform
show_tokens(text, "Xenova/gpt-4")

tokenizer_config.json:   0%|          | 0.00/460 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.01M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/917k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.23M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/98.0 [00:00<?, ?B/s]


 English  and  CAPITAL IZATION 
 � � �  � � � 
 show _tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         "
 12 . 0 * 50 = 600 
 

In [ ]:
# You need to request access before being able to use this tokenizer
show_tokens(text, "bigcode/starcoder2-15b")

tokenizer_config.json:   0%|          | 0.00/7.88k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/777k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/442k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]


 English  and  CAPITAL IZATION 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"      "  Three  tabs :  "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

In [ ]:
show_tokens(text, "facebook/galactica-1.3b")

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.14M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.00 [00:00<?, ?B/s]


 English  and  CAP ITAL IZATION 
 � � � �  � � � 
 show _ tokens  False  None  elif   ==   > =  else :  two  t abs : "      "  Three  t abs :   "         " 
 1 2 . 0 * 5 0 = 6 0 0 
 

In [ ]:
show_tokens(text, "microsoft/Phi-3-mini-4k-instruct")

 
 English and C AP IT AL IZ ATION 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"    " Three tabs : "       " 
 1 2 . 0 * 5 0 = 6 0 0 
 

# Contextualized Word Embeddings From a Language Model (Like BERT)

In [ ]:
from transformers import AutoModel, AutoTokenizer

# Load a tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load a language model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
text = 'Hello world'
tokens = tokenizer(text, return_tensors='pt')

# Process the tokens
output = model(**tokens)[0]
# or
input_ids = tokens['input_ids']
attention_mask = tokens['attention_mask']

# Running the model on the specified inputs
outputs = model(input_ids=input_ids, attention_mask=attention_mask)

# Extracting the first part of the output (token embeddings)
output = outputs[0]



In many models, such as transformers (e.g., BERT, GPT), the outputs object is often a tuple that contains several elements. When you do **outputs[0**], you are accessing the first element of this tuple, which typically represents the embeddings (or hidden states) for each token in the input sequence.

output = outputs[0]

In [ ]:
print(output)

tensor([[[-3.4816,  0.0861, -0.1819,  ..., -0.0612, -0.3911,  0.3017],
         [ 0.1898,  0.3208, -0.2315,  ...,  0.3714,  0.2478,  0.8048],
         [ 0.2071,  0.5036, -0.0485,  ...,  1.2175, -0.2292,  0.8582],
         [-3.4278,  0.0645, -0.1427,  ...,  0.0658, -0.4367,  0.3834]]],
       grad_fn=<NativeLayerNormBackward0>)


In [ ]:
# Print the keys of the 'tokens' dictionary to see the main input fields
print(tokens.keys())

dict_keys(['input_ids', 'token_type_ids', 'attention_mask'])


In [ ]:
import json

# Convert tensors to lists
tokens_converted = {key: value.tolist() for key, value in tokens.items()}

# Convert the dictionary to a formatted JSON string
formatted_json = json.dumps(tokens_converted, indent=4)

# Print the formatted JSON string
print(formatted_json)

{
    "input_ids": [
        [
            1,
            31414,
            232,
            2
        ]
    ],
    "token_type_ids": [
        [
            0,
            0,
            0,
            0
        ]
    ],
    "attention_mask": [
        [
            1,
            1,
            1,
            1
        ]
    ]
}


In [ ]:
output.shape

torch.Size([1, 4, 384])

In [ ]:
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))

[CLS]
Hello
 world
[SEP]


tensor([[[-3.4816,  0.0861, -0.1819,  ..., -0.0612, -0.3911,  0.3017],
         [ 0.1898,  0.3208, -0.2315,  ...,  0.3714,  0.2478,  0.8048],
         [ 0.2071,  0.5036, -0.0485,  ...,  1.2175, -0.2292,  0.8582],
         [-3.4278,  0.0645, -0.1427,  ...,  0.0658, -0.4367,  0.3834]]],
       grad_fn=<NativeLayerNormBackward0>)


# Text Embeddings (For Sentences and Whole Documents)

In [ ]:
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Convert text to text embeddings
vector = model.encode("Best movie ever!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vector.shape

(768,)

# Word Embeddings Beyond LLMs


In [ ]:
import gensim.downloader as api

# Load pretrained GloVe embeddings
model = api.load("glove-wiki-gigaword-50")

# Find similar words to "king"
similar_words = model.most_similar([model['king']], topn=5)
print(similar_words)

# Print each word and similarity on a new line
for word, similarity in similar_words:
    print(f"{word}: {similarity}")

[('king', 1.0000001192092896), ('prince', 0.8236179351806641), ('queen', 0.7839043140411377), ('ii', 0.7746230363845825), ('emperor', 0.7736247777938843)]
king: 1.0000001192092896
prince: 0.8236179351806641
queen: 0.7839043140411377
ii: 0.7746230363845825
emperor: 0.7736247777938843


# Recommending songs by embeddings

In [ ]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('http://millionsongdataset.com/sites/default/files/AdditionalFiles/songlist_with_mbid.txt')

# Parse the playlist dataset file. Skip the first two lines as
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlists with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns = ['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [ ]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['TRGCKLJ128F92FF90E<SEP>f26c72d3-e52c-467b-b651-679c73d8e1a7<SEP>!!!<SEP>All', 'My', 'Heroes', 'Are', 'Weirdos'] 

Playlist #2:
  ['TRRKQQU128F92FF947<SEP>f26c72d3-e52c-467b-b651-679c73d8e1a7<SEP>!!!<SEP>Bend', 'Over', 'Beethoven']


In [ ]:
from gensim.models import Word2Vec

# Train our Word2Vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4
)

In [ ]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('2849', 0.9979680776596069),
 ('2640', 0.9964019060134888),
 ('3167', 0.9963980317115784),
 ('5549', 0.9959008693695068),
 ('2715', 0.9958351850509644),
 ('3117', 0.9954560995101929),
 ('2987', 0.9953479766845703),
 ('2881', 0.9951083660125732),
 ('2886', 0.9950577616691589),
 ('3094', 0.994985044002533)]

In [ ]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [ ]:
import numpy as np

def print_recommendations(song_id):
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id),topn=5)
    )[:,0]
    return  songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
2849,Run To The Hills,Iron Maiden
2640,Red Barchetta,Rush
3167,Unchained,Van Halen
5549,November Rain,Guns N' Roses
2715,Rainbow In The Dark,Dio


In [ ]:
print_recommendations(2172)

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object
['2849' '2640' '3167' '5549' '2715']


,title,artist
id,,
2849,Run To The Hills,Iron Maiden
2640,Red Barchetta,Rush
3167,Unchained,Van Halen
5549,November Rain,Guns N' Roses
2715,Rainbow In The Dark,Dio


In [ ]:
print_recommendations(842)

title     California Love (w\/ Dr. Dre & Roger Troutman)
artist                                              2Pac
Name: 842 , dtype: object
['5668' '413' '5661' '330' '886']


,title,artist
id,,
5668,How We Do (w\/ 50 Cent),The Game
413,If I Ruled The World (Imagine That) (w\/ Laury...,Nas
5661,Sweet Dreams,Beyonce
330,Hate It Or Love It (w\/ 50 Cent),The Game
886,Heartless,Kanye West
